In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated,List
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_ollama import ChatOllama,OllamaEmbeddings
from langchain_community.document_loaders import PyPDFLoader,DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langgraph.prebuilt import ToolNode,tools_condition
from langchain_core.tools import tool
from langchain_community.vectorstores import FAISS
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from langchain_community.tools.tavily_search import TavilySearchResults
from pydantic import BaseModel
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
import os
import operator
load_dotenv()

In [ ]:
# Configuration
MODEL_NAME = "kimi-k2-thinking:cloud"
EMBEDDING_MODEL = "nomic-embed-text:latest"
PDF_PATH = r"C:\Users\miyas\Downloads"

In [ ]:
def load_pdfs(pdf_path):
    loader = DirectoryLoader(pdf_path, glob="**/*.pdf", show_progress=True, use_multithreading=True, silent_errors=True)
    return loader.load()

In [ ]:
loaded_docs = load_pdfs(PDF_PATH)
loaded_docs

In [ ]:
len(loaded_docs)

In [ ]:
def chunk_documents(docs: list[Document], chunk_size: int = 1000, chunk_overlap: int = 400) -> list[Document]:
    """Split documents into smaller chunks for embedding and retrieval."""
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    return text_splitter.split_documents(docs)

In [ ]:
chunked_docs = chunk_documents(loaded_docs)
len(chunked_docs)

In [ ]:
embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)

In [ ]:
vectorstore = FAISS.from_documents(chunked_docs, embeddings)

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 15,"search_type":"similarity"})

In [ ]:
model=ChatOllama(model=MODEL_NAME,
                reasoning=True
                 )

In [ ]:
# --------------------------------------------------
# Graph State
# --------------------------------------------------
class State(TypedDict):
    question: str
    need_retrieval: bool

    docs: List[Document]

    answer: str

In [ ]:
class RetrieveDecision(BaseModel):
    should_retrieve: bool = Field(
        ...,
        description="True if external documents are needed to answer reliably, else False."
    )

decide_retrieval_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You decide whether retrieval is needed.\n"
            "Return JSON that matches this schema:\n"
            "{{'should_retrieve': boolean}}\n\n"
            "Guidelines:\n"
            "- should_retrieve=True if answering requires specific facts, citations, or info likely not in the model.\n"
            "- should_retrieve=False for general explanations, definitions, or reasoning that doesn't need sources.\n"
            "- If unsure, choose True."
        ),
        ("human", "Question: {question}"),
    ]
)


# IMPORTANT: no `.content` for structured output
should_retrieve_model = model.with_structured_output(RetrieveDecision)

def decide_retrieval(state: "State"):
    decision: RetrieveDecision = should_retrieve_model.invoke(
        decide_retrieval_prompt.format_messages(question=state["question"])
    )
    return {"need_retrieval": decision.should_retrieve}

In [ ]:
direct_generation_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Answer the question using only your general knowledge.\n"
            "Do NOT assume access to external documents.\n"
            "If you are unsure or the answer requires specific sources, say:\n"
            "'I don't know based on my general knowledge.'"
        ),
        ("human", "{question}"),
    ]
)


def generate_direct(state: State):
    out = model.invoke(
        direct_generation_prompt.format_messages(
            question=state["question"]
        )
    )
    return {
        "answer": out.content
    }

In [ ]:
def retrieve(state: State):
    return {"docs": retriever.invoke(state["question"])}

In [ ]:
def route_after_decide(state: State) -> Literal["generate_direct", "retrieve"]:
    if state["need_retrieval"]:
        return "retrieve"
    return "generate_direct"

In [ ]:
def generate_with_retrieval(state: State):
    # For simplicity, just return the retrieved docs as the "answer"
    # In a real system, you'd have another prompt that combines the question + retrieved docs to generate an answer
    prompt=PromptTemplate.from_template(
        "Question: {question}\n\nRetrieved Docs:\n{docs}\n\nAnswer the question based on the retrieved docs.Also give refrences to the docs in your answer. If the docs don't contain the answer, say you don't know."
    )
    
    response=model.invoke(
        prompt.format(question=state["question"], docs="\n".join([doc.page_content for doc in state["docs"]]))
    )
    
    
    return {"answer": response.content}

In [ ]:
g = StateGraph(State)

# --------------------
# Nodes
# --------------------
g.add_node("decide_retrieval", decide_retrieval)
g.add_node("generate_direct", generate_direct)
g.add_node("retrieve", retrieve)
g.add_node("generate_with_retrieval", generate_with_retrieval)
# --------------------
# Edges
# --------------------
g.add_edge(START, "decide_retrieval")

g.add_conditional_edges(
    "decide_retrieval",
    route_after_decide,
    {
        "generate_direct": "generate_direct",
        "retrieve": "retrieve",
    },
)

g.add_edge("generate_direct", END)
g.add_edge("retrieve", "generate_with_retrieval")
g.add_edge("generate_with_retrieval", END)
app = g.compile()
app

In [ ]:
result = app.invoke(
    {
        "question": "LLM councils",
        "need_retrieval": False,
        "docs": [],
        "answer": "",
    }
)
print("Answer:", result["answer"])
print("References:")
for doc in result["docs"]:
    print(doc.page_content[:300],"...")